# 01 - Data Preparation

Downloads the substation semantic segmentation dataset archive from Zenodo.

- **Dataset:** A Semantically Annotated 15-Class Ground Truth Dataset for Substation Equipment
- **Record:** https://zenodo.org/records/7884270 (DOI: [10.5281/zenodo.7884270](https://doi.org/10.5281/zenodo.7884270))
- **File:** `substation-semantic-dataset.zip` (about 1.84 GB, 1660 images, 15 classes)
- **License:** CC BY 4.0

The notebook runs both locally and in Google Colab:

| Environment | Archive is saved to |
|---|---|
| Local | `<repo>/data/raw/substation-semantic-dataset.zip` |
| Google Colab | `/content/drive/MyDrive/substation-semantic-dataset.zip` (where `02_substation_Semantic_Segmentation_EDA_V1.ipynb` looks for it) |

The download is resumable and is skipped when a verified copy already exists.

## 1. Configuration

In [ ]:
import hashlib
import zipfile
from pathlib import Path

import requests
from tqdm.auto import tqdm

DATASET_URL = "https://zenodo.org/records/7884270/files/substation-semantic-dataset.zip?download=1"
DATASET_ARCHIVE_NAME = "substation-semantic-dataset.zip"

# Published on the Zenodo record; used to verify the download.
EXPECTED_SIZE_BYTES = 1_844_470_522
EXPECTED_MD5 = "ca897cb85a3b944af6a1355e83530c32"

CHUNK_SIZE = 1024 * 1024  # 1 MB

## 2. Resolve the download directory

In [ ]:
try:
    from google.colab import drive

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    download_directory = Path("/content/drive/MyDrive")
else:
    # The notebook lives in <repo>/notebooks, so the repo root is one level up.
    repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    download_directory = repo_root / "data" / "raw"

download_directory.mkdir(parents=True, exist_ok=True)
archive_path = download_directory / DATASET_ARCHIVE_NAME

print("Running in Colab:", IN_COLAB)
print("Archive path:", archive_path)

## 3. Download the archive

In [ ]:
def md5_of_file(file_path, chunk_size=CHUNK_SIZE):
    """Return the MD5 hex digest of a file, reading it in chunks."""
    digest = hashlib.md5()
    with open(file_path, "rb") as file_handle, tqdm(
        total=file_path.stat().st_size, unit="B", unit_scale=True, desc="Verifying MD5"
    ) as progress:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
            progress.update(len(chunk))
    return digest.hexdigest()


def download_file(url, destination, expected_size=None, chunk_size=CHUNK_SIZE):
    """Stream a file to disk, resuming a partial download when possible."""
    partial_path = destination.with_name(destination.name + ".part")
    resume_from = partial_path.stat().st_size if partial_path.exists() else 0

    if expected_size is not None and resume_from > expected_size:
        partial_path.unlink()
        resume_from = 0

    if expected_size is None or resume_from < expected_size:
        headers = {"Range": f"bytes={resume_from}-"} if resume_from else {}
        with requests.get(url, headers=headers, stream=True, timeout=60) as response:
            response.raise_for_status()

            # 206 means the server honoured the Range request; otherwise start over.
            if resume_from and response.status_code != 206:
                resume_from = 0
            remaining = int(response.headers.get("Content-Length", 0))
            total = resume_from + remaining if remaining else expected_size

            with open(partial_path, "ab" if resume_from else "wb") as file_handle, tqdm(
                total=total, initial=resume_from, unit="B", unit_scale=True, desc=destination.name
            ) as progress:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    file_handle.write(chunk)
                    progress.update(len(chunk))

    partial_path.replace(destination)
    return destination


if archive_path.exists() and archive_path.stat().st_size == EXPECTED_SIZE_BYTES:
    print("Archive already downloaded, skipping:", archive_path)
else:
    if archive_path.exists():
        print("Existing archive has an unexpected size, downloading again.")
        archive_path.unlink()
    download_file(DATASET_URL, archive_path, expected_size=EXPECTED_SIZE_BYTES)
    print("Download complete:", archive_path)

## 4. Verify the archive

In [ ]:
actual_size = archive_path.stat().st_size
if actual_size != EXPECTED_SIZE_BYTES:
    raise ValueError(f"Size mismatch: expected {EXPECTED_SIZE_BYTES:,} bytes, got {actual_size:,} bytes")

actual_md5 = md5_of_file(archive_path)
if actual_md5 != EXPECTED_MD5:
    raise ValueError(f"MD5 mismatch: expected {EXPECTED_MD5}, got {actual_md5}")

if not zipfile.is_zipfile(archive_path):
    raise ValueError(f"Not a valid zip archive: {archive_path}")

print(f"Size : {actual_size:,} bytes ({actual_size / 1024**3:.2f} GB)")
print(f"MD5  : {actual_md5} (matches Zenodo)")

## 5. Preview the archive contents

In [ ]:
with zipfile.ZipFile(archive_path) as archive:
    archive_entries = archive.infolist()

file_entries = [entry for entry in archive_entries if not entry.is_dir()]
print(f"Entries in archive: {len(archive_entries):,} ({len(file_entries):,} files)")
print("First entries:")
for entry in archive_entries[:10]:
    print(f"  {entry.filename}  ({entry.file_size:,} bytes)")

## Next step

Continue with `02_substation_Semantic_Segmentation_EDA_V1.ipynb`, which extracts the archive and runs the exploratory data analysis.